## 1. Identificação da Fonte de Dados

- Fonte: "Demonstrativo Fecap v3.csv", fornecido pela CTI Global.
- Formato: CSV, delimitador ";", encoding Latin-1, sem cabeçalho.
- Estrutura esperada: 4 colunas — ano, cenário, conta, valor.

In [ ]:
import pandas as pd
import csv

df_bruto = pd.read_csv(
    "Demonstrativo Fecap v3.csv",
    sep=";",
    header=None,
    names=["ano", "cenario", "conta", "valor"],
    encoding="latin-1",
    dtype=str,
    quoting=csv.QUOTE_ALL,
    keep_default_na=False,
)

print(f"Linhas carregadas: {len(df_bruto):,}")
df_bruto.head()


Linhas carregadas: 1,002,000


,ano,cenario,conta,valor
0,Ano 1,Total Cen_00001,,"0,025138815"
1,Ano 1,Total Cen_00001,BAL - Total do Ativo,"10.631.127.075,2579"
2,Ano 1,Total Cen_00001,BAL - Ativo Circulante,"1.795.720.872,17537"
3,Ano 1,Total Cen_00001,BAL - Disponível,"1.205.264.965,51818"
4,Ano 1,Total Cen_00001,BAL - Contas a Receber - SWAP,"95.680.421,9232409"


## 2. Coleta de Dados

O arquivo foi carregado em sua forma bruta, sem nenhuma transformação de
conteúdo, preservando a camada "raw" exigida pelo escopo do projeto. Os
parâmetros de leitura foram definidos com base no formato conhecido do
arquivo fonte:

- `sep=";"` — delimitador usado no CSV.
- `header=None` e `names=[...]` — o arquivo não possui cabeçalho.
- `encoding="latin-1"` — necessário para caracteres acentuados do arquivo original.
- `dtype=str` — todas as colunas mantidas como texto nesta etapa, evitando
  perda de precisão antes da limpeza (ex.: números no formato brasileiro
  ainda não convertidos).
- `quoting=csv.QUOTE_ALL` — os campos do arquivo vêm entre aspas.
- `keep_default_na=False` — impede que strings vazias sejam interpretadas
  automaticamente como valores ausentes (NaN), já que uma "conta" vazia
  tem significado próprio neste dataset (ver Seleção de Dados).

A coleta resultou em 1.002.000 linhas e 4 colunas, confirmando o volume
esperado do arquivo fonte.

In [ ]:
print("Tipos de dado por coluna:")
print(df_bruto.dtypes)
print()
print("Valores distintos de 'ano':", df_bruto["ano"].nunique())
print("Valores distintos de 'cenario':", df_bruto["cenario"].nunique())
print("Valores distintos de 'conta':", df_bruto["conta"].nunique())

Tipos de dado por coluna:
ano        object
cenario    object
conta      object
valor      object
dtype: object

Valores distintos de 'ano': 12
Valores distintos de 'cenario': 1200
Valores distintos de 'conta': 70


In [ ]:
n_conta_vazia = (df_bruto["conta"] == "").sum()
n_valor_vazio = (df_bruto["valor"].str.strip() == "").sum()

print(f"Linhas com 'conta' vazia: {n_conta_vazia:,}")
print(f"Linhas com 'valor' vazio: {n_valor_vazio:,}")

Linhas com 'conta' vazia: 43,200
Linhas com 'valor' vazio: 0


In [ ]:
contas_por_ano = (
    df_bruto[df_bruto["conta"] != ""]
    .groupby("ano")["conta"]
    .nunique()
    .sort_index()
)
contas_por_ano

,conta
ano,
Ano 1,68
Ano 10,69
Ano 11,69
Ano 12,42
Ano 2,68
Ano 3,69
Ano 4,69
Ano 5,69
Ano 6,69


## 3. Descrição e Exploração dos Dados

A base carregada contém 1.002.000 linhas e 4 colunas (ano, cenário, conta,
valor), todas inicialmente como texto. Foram identificadas 12 categorias de
"ano" e 1.200 categorias de "cenário", com 69 nomes de conta distintos
(além da linha com "conta" vazia, usada para peso de cenário e separadores
estruturais).

**Achados de qualidade:**
- 43.200 linhas possuem a coluna "conta" vazia — não são inconsistências,
  e sim linhas estruturais (peso do cenário ou separador de bloco),
  tratadas separadamente na etapa de Seleção.
- Nenhuma linha possui "valor" ausente/vazio.
- O número de contas por ano **não é uniforme**: Ano 1 e Ano 2 têm 68
  contas, Anos 3 a 11 têm 69, e o **Ano 12 tem apenas 42 contas** — uma
  redução estrutural relevante, possivelmente relacionada ao fim do
  horizonte de concessão. Este achado será documentado como limitação
  conhecida e sinalizado para validação com a CTI.

In [ ]:
conta_vazia = df_bruto["conta"] == ""
eh_zero = df_bruto["valor"].str.strip() == "0"

separadores = df_bruto.loc[conta_vazia & eh_zero].copy()
pesos = df_bruto.loc[conta_vazia & ~eh_zero].copy()
dados = df_bruto.loc[~conta_vazia].copy()

print(f"Total bruto:   {len(df_bruto):,}")
print(f"dados:         {len(dados):,}  <- isso é o que selecionamos para a análise")
print(f"pesos:         {len(pesos):,}  <- guardamos, mas não é 'conta'")
print(f"separadores:   {len(separadores):,}  <- descartamos, é só marcador estrutural")

Total bruto:   1,002,000
dados:         958,800  <- isso é o que selecionamos para a análise
pesos:         14,400  <- guardamos, mas não é 'conta'
separadores:   28,800  <- descartamos, é só marcador estrutural


## 4. Seleção de Dados

Da base bruta, foram selecionadas para a análise apenas as linhas com a
coluna "conta" preenchida (dados contábeis reais dos demonstrativos
BAL/DRE/FLU). Linhas de "separadores" (conta vazia, valor "0") foram
excluídas por serem apenas marcadores estruturais de fim de bloco.
Linhas de "pesos" (conta vazia, valor diferente de "0") foram mantidas
separadamente para reintegração posterior como peso do cenário.

In [ ]:
def texto_para_float(valor_texto):
    if valor_texto is None or valor_texto.strip() == "":
        return float("nan")
    v = valor_texto.strip().replace(".", "").replace(",", ".")
    return float(v)

In [ ]:
dados["valor_num"] = dados["valor"].apply(texto_para_float)
dados["conta_normalizada"] = dados["conta"].str.replace(r"\s+", " ", regex=True).str.strip()

pesos["peso_cenario"] = pesos["valor"].apply(texto_para_float)

dados[["conta", "conta_normalizada", "valor_num"]].head()

,conta,conta_normalizada,valor_num
1,BAL - Total do Ativo,BAL - Total do Ativo,1.063113e+10
2,BAL - Ativo Circulante,BAL - Ativo Circulante,1.795721e+09
3,BAL - Disponível,BAL - Disponível,1.205265e+09
4,BAL - Contas a Receber - SWAP,BAL - Contas a Receber - SWAP,9.568042e+07
5,BAL - Contas a Receber - Partes Relacionadas,BAL - Contas a Receber - Partes Relacionadas,5.846615e+05


## 5. Limpeza e Uniformização

Os valores da coluna "valor" (originalmente texto em formato numérico
brasileiro, ex: "10.631.127.075,2579") foram convertidos para float via
função dedicada. Os nomes de conta foram normalizados, removendo espaços
duplicados que ocorriam em alguns registros originais (ex.: "Emprést" e
"Outros deb", nomes truncados na fonte, preservados como estão por não
haver forma de recuperar o nome completo sem contato com a CTI).

In [ ]:
dados["ano_num"] = dados["ano"].str.extract(r"(\d+)").astype(int)
dados["demonstrativo"] = dados["conta"].str.split(" - ", n=1).str[0].str.strip()

dados[["ano", "ano_num", "conta", "demonstrativo"]].head()

,ano,ano_num,conta,demonstrativo
1,Ano 1,1,BAL - Total do Ativo,BAL
2,Ano 1,1,BAL - Ativo Circulante,BAL
3,Ano 1,1,BAL - Disponível,BAL
4,Ano 1,1,BAL - Contas a Receber - SWAP,BAL
5,Ano 1,1,BAL - Contas a Receber - Partes Relacionadas,BAL


In [ ]:
pesos_para_merge = pesos[["ano", "cenario", "peso_cenario"]]

base_integrada = dados.merge(pesos_para_merge, on=["ano", "cenario"], how="left")
base_integrada["cenario_num"] = base_integrada["cenario"].str.extract(r"(\d+)").astype(int)

base_integrada.head()

,ano,cenario,conta,valor,valor_num,conta_normalizada,ano_num,demonstrativo,peso_cenario,cenario_num
0,Ano 1,Total Cen_00001,BAL - Total do Ativo,"10.631.127.075,2579",1.063113e+10,BAL - Total do Ativo,1,BAL,0.025139,1
1,Ano 1,Total Cen_00001,BAL - Ativo Circulante,"1.795.720.872,17537",1.795721e+09,BAL - Ativo Circulante,1,BAL,0.025139,1
2,Ano 1,Total Cen_00001,BAL - Disponível,"1.205.264.965,51818",1.205265e+09,BAL - Disponível,1,BAL,0.025139,1
3,Ano 1,Total Cen_00001,BAL - Contas a Receber - SWAP,"95.680.421,9232409",9.568042e+07,BAL - Contas a Receber - SWAP,1,BAL,0.025139,1
4,Ano 1,Total Cen_00001,BAL - Contas a Receber - Partes Relacionadas,"584.661,46688296",5.846615e+05,BAL - Contas a Receber - Partes Relacionadas,1,BAL,0.025139,1


## 6. Derivação e Integração

Foram derivados os atributos "ano_num" (inteiro extraído de "Ano N"),
"demonstrativo" (BAL/DRE/FLU extraído do prefixo da conta) e
"conta_normalizada" (espaços duplicados uniformizados). A integração
combina a camada de dados contábeis com a camada de pesos de cenário,
via chave (ano, cenário), adicionando o atributo "peso_cenario" a cada
linha de dado real.

In [ ]:
base_final = base_integrada[
    ["ano", "ano_num", "cenario", "cenario_num", "peso_cenario",
     "demonstrativo", "conta", "conta_normalizada", "valor_num"]
].rename(columns={"valor_num": "valor"})

base_final["ano"] = base_final["ano"].astype("category")
base_final["demonstrativo"] = base_final["demonstrativo"].astype("category")

base_final = base_final.sort_values(
    ["ano_num", "cenario_num", "demonstrativo", "conta"]
).reset_index(drop=True)

base_final.head()

,ano,ano_num,cenario,cenario_num,peso_cenario,demonstrativo,conta,conta_normalizada,valor
0,Ano 1,1,Total Cen_00001,1,0.025139,BAL,BAL - Capital Social,BAL - Capital Social,-3.096189e+08
1,Ano 1,1,Total Cen_00001,1,0.025139,BAL,BAL - Patrimônio Líquido,BAL - Patrimônio Líquido,-2.014435e+09
2,Ano 1,1,Total Cen_00001,1,0.025139,BAL,BAL - Prov para Contingências,BAL - Prov para Contingências,-3.850246e+07
3,Ano 1,1,Total Cen_00001,1,0.025139,BAL,BAL - Reserva de Retenção de Lucros,BAL - Reserva de Retenção de Lucros,-3.143802e+09
4,Ano 1,1,Total Cen_00001,1,0.025139,BAL,BAL - Reservas Legais,BAL - Reservas Legais,-6.192379e+07


## 7. Formatação

A base final foi tipada (colunas "ano" e "demonstrativo" como category,
para otimizar memória) e ordenada por ano, cenário, demonstrativo e conta,
deixando-a pronta para consumo pelas próximas etapas (análise descritiva
e definição de KPIs financeiros).

## 8. Verificação de Qualidade Final

Antes de considerar o processo concluído, foram feitas cinco verificações
automáticas para confirmar que nada se perdeu ou ficou incorreto ao longo
do processo.


In [ ]:
# Teste 1: conservação de linhas (nada se perdeu ou duplicou sem explicação)
assert len(dados) + len(pesos) + len(separadores) == len(df_bruto), \
    "ERRO: a soma das camadas não bate com o total bruto"
print("✅ Teste 1 OK — conservação de linhas")

# Teste 2: base final não tem valores nulos inesperados
assert base_final["valor"].isna().sum() == 0, \
    "ERRO: existem valores nulos na coluna 'valor' da base final"
print("✅ Teste 2 OK — sem valores nulos em 'valor'")

# Teste 3: todo peso_cenario foi encontrado (merge não deixou buraco)
n_sem_peso = base_final["peso_cenario"].isna().sum()
print(f"Linhas sem peso_cenario após o merge: {n_sem_peso:,}")
assert n_sem_peso == 0, "ERRO: existem linhas sem peso de cenário associado"
print("✅ Teste 3 OK — merge de pesos completo")

# Teste 4: nenhuma linha de conta vazia vazou pra base final (a seleção funcionou)
assert (base_final["conta"] == "").sum() == 0, \
    "ERRO: linhas de conta vazia vazaram para a base final"
print("✅ Teste 4 OK — seleção de dados correta")

# Teste 5: número de anos e cenários bate com o esperado
assert base_final["ano"].nunique() == 12, f"ERRO: esperado 12 anos, encontrado {base_final['ano'].nunique()}"
assert base_final["cenario_num"].nunique() == 1200, f"ERRO: esperado 1200 cenários, encontrado {base_final['cenario_num'].nunique()}"
print("✅ Teste 5 OK — 12 anos e 1.200 cenários confirmados")

print("\n🎉 Todos os testes de sanidade passaram.")

✅ Teste 1 OK — conservação de linhas
✅ Teste 2 OK — sem valores nulos em 'valor'
Linhas sem peso_cenario após o merge: 0
✅ Teste 3 OK — merge de pesos completo
✅ Teste 4 OK — seleção de dados correta
✅ Teste 5 OK — 12 anos e 1.200 cenários confirmados

🎉 Todos os testes de sanidade passaram.


## Organização da Planilha para a Equipe

As oito etapas exigidas pelo projeto terminam na seção anterior. A partir
daqui, o notebook monta a base final em formato de planilha (uma linha
por cenário e ano, com uma coluna para cada conta), que é o arquivo usado
pelo restante da equipe para trabalhar em Excel.


In [ ]:
import gc

del df_bruto, dados, pesos, separadores, base_integrada
gc.collect()

base_planilha = base_final.pivot(
    index=["cenario_num", "ano"],
    columns="conta_normalizada",
    values="valor"
).reset_index()

# Ordena as linhas por ano e depois por número do cenário
base_planilha = base_planilha.sort_values(["ano", "cenario_num"]).reset_index(drop=True)

# Cria o identificador legível do cenário só na planilha final (não no base_final)
base_planilha.insert(0, "Cenário", "Cenário " + base_planilha["cenario_num"].astype(str))
base_planilha = base_planilha.drop(columns=["cenario_num"])
base_planilha = base_planilha.rename(columns={"ano": "Ano"})

# Ordena as colunas: primeiro as fixas, depois as contas em ordem alfabética
# (isso já agrupa BAL, DRE e FLU automaticamente, por causa do prefixo)
colunas_fixas = ["Cenário", "Ano"]
colunas_contas = sorted(c for c in base_planilha.columns if c not in colunas_fixas)
base_planilha = base_planilha[colunas_fixas + colunas_contas]

print(f"Linhas: {len(base_planilha):,}")
print(f"Colunas: {len(base_planilha.columns):,}")
base_planilha.head()


Linhas: 14,400
Colunas: 71


conta_normalizada,Cenário,Ano,BAL - Amortização - Intangível,BAL - Amortização Acumulada,BAL - At Fiscal Diferido,BAL - Ativo Circulante,BAL - Capital Social,BAL - Contas a Pagar - Parte Relacionada,BAL - Contas a Receber - Clientes,BAL - Contas a Receber - Partes Relacionadas,...,FLU - Distribuição para Acionista,FLU - Entradas,FLU - Geração de Caixa,FLU - Imposto de Renda e Contribuição Social,FLU - Investimentos,FLU - Receita,FLU - Resultado Financeiro,FLU - Saldo Final,FLU - Saldo Inicial,FLU - Tributos
0,Cenário 1,Ano 1,-5.047124e+09,-3.011128e+07,-1.039693e+08,1.795721e+09,-309618939.0,-1.408627e+07,4.538710e+08,584661.466883,...,-1.532408e+09,9.162700e+07,4.306498e+08,-9.344728e+08,-5.340012e+08,5.396831e+09,-7.304044e+08,1.205265e+09,7.746152e+08,-4.732576e+08
1,Cenário 2,Ano 1,-5.047124e+09,-2.998150e+07,-1.036850e+08,1.789812e+09,-309618939.0,-1.408627e+07,4.538710e+08,584661.466883,...,-1.532408e+09,9.434815e+07,4.243157e+08,-9.312567e+08,-5.322824e+08,5.377709e+09,-7.256803e+08,1.198931e+09,7.746152e+08,-4.718228e+08
2,Cenário 3,Ano 1,-5.047124e+09,-2.983733e+07,-1.036096e+08,1.790756e+09,-309618939.0,-1.408627e+07,4.538710e+08,584661.466883,...,-1.532408e+09,9.737124e+07,4.211763e+08,-9.304040e+08,-5.303729e+08,5.350463e+09,-7.083835e+08,1.195792e+09,7.746152e+08,-4.697783e+08
3,Cenário 4,Ano 1,-5.047124e+09,-2.980375e+07,-1.036349e+08,1.791311e+09,-309618939.0,-1.408627e+07,4.538710e+08,584661.466883,...,-1.532408e+09,9.807547e+07,4.216127e+08,-9.306903e+08,-5.299281e+08,5.346725e+09,-7.050620e+08,1.196228e+09,7.746152e+08,-4.694978e+08
4,Cenário 5,Ano 1,-5.047124e+09,-2.967476e+07,-1.028104e+08,1.770574e+09,-309618939.0,-1.408627e+07,4.538710e+08,584661.466883,...,-1.532408e+09,9.759232e+07,4.052842e+08,-9.213635e+08,-5.282197e+08,5.334754e+09,-7.231589e+08,1.179899e+09,7.746152e+08,-4.685996e+08


In [ ]:
base_planilha.to_csv("base_analitica_planilha.csv", index=False, encoding="utf-8-sig")

print("Exportado com sucesso.")
print(f"Linhas na planilha: {len(base_planilha):,}")


Exportado com sucesso.
Linhas na planilha: 14,400


In [ ]:
from google.colab import files
files.download("base_analitica_planilha.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>